# 🦅 FALCON CFE — Entrenamiento de IA satelital en Google Colab

Este notebook entrena el detector de infraestructura de CFE usando **GPU gratis** de Google.

**IMPORTANTE — activa la GPU primero:**
1. Menú **Entorno de ejecución** → **Cambiar tipo de entorno de ejecución**
2. En *Acelerador por hardware* elige **GPU (T4)** → Guardar

Luego corre las celdas en orden (botón ▶ de cada una).

## 1. Verificar que hay GPU

In [ ]:
!nvidia-smi

## 2. Clonar el repositorio

In [ ]:
!git clone https://github.com/Prime119/ProyectoMEC.git
%cd ProyectoMEC

## 3. Instalar dependencias

In [ ]:
!pip install -q pillow httpx ultralytics

## 4. Preparar dataset — hasta 2000 imágenes auto-etiquetadas desde OSM + satélite

Esto descarga imágenes satelitales de las ciudades del Noroeste y genera las etiquetas automáticamente. Puede tardar 20-40 min (está bajando muchas imágenes).

In [ ]:
!python entrenamiento/preparar_dataset.py --zoom 18 --radio 200 --max-imagenes 2000

## 5. Generar la configuración del dataset (24 clases)

In [ ]:
!python entrenamiento/generar_config.py

## 6. Entrenar el modelo (YOLOv8m en GPU)

Con GPU T4 y ~2000 imágenes, 100 épocas tardan aprox. 1-2 horas. Al terminar exporta el modelo a ONNX automáticamente.

In [ ]:
!python entrenamiento/entrenar.py --modelo yolov8m.pt --epochs 100 --imgsz 640 --device 0

## 7. Descargar el modelo entrenado (best.onnx)

Guarda este archivo. Luego en tu PC:
```
set PALANTIR_MODELO_ONNX=C:\ruta\a\cfe_satelital.onnx
python falcon.py
```

In [ ]:
import glob, shutil
from google.colab import files
onnx = glob.glob('runs/detect/**/best.onnx', recursive=True)
if onnx:
    print('Modelo encontrado:', onnx[0])
    shutil.copy(onnx[0], 'cfe_satelital.onnx')
    files.download('cfe_satelital.onnx')
else:
    print('No se encontro best.onnx. Revisa que el entrenamiento haya terminado bien.')

## (Opcional) Ver resultados del entrenamiento

Métricas, curvas y ejemplos de detección.

In [ ]:
import glob
from IPython.display import Image, display
for img in glob.glob('runs/detect/**/results.png', recursive=True):
    display(Image(filename=img))
for img in glob.glob('runs/detect/**/val_batch0_pred.jpg', recursive=True)[:2]:
    display(Image(filename=img))